In [ ]:
import pandas as pd
import numpy as np
path = '../data/raw'
products = pd.read_csv(f'{path}/products.csv')
customers = pd.read_csv(f'{path}/customers.csv')
promotions = pd.read_csv(f'{path}/promotions.csv')
geography = pd.read_csv(f'{path}/geography.csv')
orders = pd.read_csv(f'{path}/orders.csv')
order_items = pd.read_csv(f'{path}/order_items.csv')
payments = pd.read_csv(f'{path}/payments.csv')
shipments = pd.read_csv(f'{path}/shipments.csv')
returns = pd.read_csv(f'{path}/returns.csv')
reviews = pd.read_csv(f'{path}/reviews.csv')
web_traffic = pd.read_csv(f'{path}/web_traffic.csv')

# parse date nếu cần
for df, cols in [
    (orders, ["order_date"]),
    (customers, ["signup_date"]),
    (shipments, ["ship_date", "delivery_date"]),
    (returns, ["return_date"]),
    (reviews, ["review_date"]),
    (web_traffic, ["date"]),
]:
    for c in cols:
        if c in df.columns:
            df[c] = pd.to_datetime(df[c], errors="coerce")

C:\Users\admin\AppData\Local\Temp\ipykernel_11688\450133163.py:9: DtypeWarning: Columns (6) have mixed types. Specify dtype option on import or set low_memory=False.
  order_items = pd.read_csv(f'{path}/order_items.csv')


In [3]:
# q1
orders_sorted = orders.sort_values(["customer_id", "order_date"]).copy()
orders_sorted["prev_order_date"] = orders_sorted.groupby("customer_id")["order_date"].shift(1)
orders_sorted["gap_days"] = (
    orders_sorted["order_date"] - orders_sorted["prev_order_date"]
).dt.days

gaps = orders_sorted["gap_days"].dropna()
median_gap = gaps.median()

print("Q1 median gap:", median_gap)

Q1 median gap: 144.0


In [4]:
# Q2
df = products.dropna(subset=["segment", "price", "cogs"]).copy()
df = df[df["price"] > 0]
df["gross_margin"] = (df["price"] - df["cogs"]) / df["price"]

q2 = df.groupby("segment")["gross_margin"].mean().sort_values(ascending=False)
print(q2)
print("Q2 answer:", q2.idxmax())

segment
Standard       0.313442
Premium        0.285377
All-weather    0.284176
Activewear     0.265600
Performance    0.263650
Balanced       0.258038
Trendy         0.240758
Everyday       0.236343
Name: gross_margin, dtype: float64
Q2 answer: Standard


In [5]:
# Q3
q3_df = returns.merge(
    products[["product_id", "category"]],
    on="product_id",
    how="inner"
)

q3 = (
    q3_df[q3_df["category"] == "Streetwear"]["return_reason"]
    .value_counts()
)

print(q3)
print("Q3 answer:", q3.idxmax())

return_reason
wrong_size          7626
defective           4330
not_as_described    3854
changed_mind        3830
late_delivery       2159
Name: count, dtype: int64
Q3 answer: wrong_size


In [6]:
# Q4
q4 = (
    web_traffic.dropna(subset=["traffic_source", "bounce_rate"])
    .groupby("traffic_source")["bounce_rate"]
    .mean()
    .sort_values()
)

print(q4)
print("Q4 answer:", q4.idxmin())

traffic_source
email_campaign    0.004458
social_media      0.004476
paid_search       0.004478
referral          0.004499
organic_search    0.004504
direct            0.004511
Name: bounce_rate, dtype: float64
Q4 answer: email_campaign


In [7]:
# Q5
q5_pct = order_items["promo_id"].notna().mean() * 100
print("Q5 percent:", q5_pct)

Q5 percent: 38.663493169565214


In [8]:
# Q6
cust_nonnull = customers.dropna(subset=["age_group"]).copy()

q6_df = cust_nonnull[["customer_id", "age_group"]].merge(
    orders[["order_id", "customer_id"]],
    on="customer_id",
    how="left"
)

orders_per_group = q6_df.groupby("age_group")["order_id"].count()
customers_per_group = cust_nonnull.groupby("age_group")["customer_id"].nunique()

q6 = (orders_per_group / customers_per_group).sort_values(ascending=False)

print(q6)
print("Q6 answer:", q6.idxmax())

age_group
55+      5.406851
45-54    5.357241
35-44    5.337343
25-34    5.245226
18-24    5.226656
dtype: float64
Q6 answer: 55+


In [13]:
# Q7
q7_df = (
    order_items.assign(line_revenue=order_items["quantity"] * order_items["unit_price"])
    .merge(orders[["order_id", "zip"]], on="order_id", how="left")
    .merge(geography[["zip", "region"]], on="zip", how="left")
)

q7 = q7_df.groupby("region")["line_revenue"].sum().sort_values(ascending=False)

print(q7)
print("Q7 answer:", q7.idxmax())

region
East       7.637533e+09
Central    4.941908e+09
West       3.851035e+09
Name: line_revenue, dtype: float64
Q7 answer: East


In [10]:
# Q8
q8 = (
    orders.loc[orders["order_status"] == "cancelled", "payment_method"]
    .value_counts()
)

print(q8)
print("Q8 answer:", q8.idxmax())

payment_method
credit_card      28452
cod              15468
paypal            7817
apple_pay         5190
bank_transfer     2535
Name: count, dtype: int64
Q8 answer: credit_card


In [ ]:
# Q9
# tỷ lệ trả hàng cao nhất = số bản ghi trong returns / số dòng trong order_items


# số bản ghi trong returns
ret_size = (
    returns.merge(products[["product_id", "size"]], on="product_id", how="inner")
    .groupby("size")
    .size()
)

# số dòng trong order_items
oi_size = (
    order_items.merge(products[["product_id", "size"]], on="product_id", how="inner")
    .groupby("size")
    .size()
)

q9 = (ret_size / oi_size).sort_values(ascending=False)


q9 = q9.loc[[s for s in ["S", "M", "L", "XL"] if s in q9.index]]

print(q9)
print("Q9 answer:", q9.idxmax())

size
S     0.056515
M     0.055660
L     0.056250
XL    0.055200
dtype: float64
Q9 answer: S


In [ ]:
# Q10
q10 = (
    payments.groupby("installments")["payment_value"]
    .mean()
    .sort_values(ascending=False)
)


q10 = q10.loc[[i for i in [1, 3, 6, 12] if i in q10.index]]

print(q10)
print("Q10 answer:", q10.idxmax())

installments
1     24113.274166
3     24399.635486
6     24446.654403
12    24245.772694
Name: payment_value, dtype: float64
Q10 answer: 6


# ĐÁP ÁN:
|Q NUMBER| ANS|
|:---:|:---:|
|1|C|
|2|D|
|3|B|
|4|C|
|5|C|
|6|A|
|7|C|
|8|A|
|9|A|
|10|C|